# SatQuery AI — Division 2: Qwen2.5-VL 4-Bit QLoRA Remote-Sensing Adaptation

This notebook orchestrates the complete, authentic QLoRA instruction tuning of **Qwen2.5-VL** (`Qwen/Qwen2.5-VL-3B-Instruct`) on Google Colab CUDA hardware.

### Pipeline Architecture
- **Source of Truth**: All business logic, dataset collation, SAR preprocessing, model architecture guards, and evaluation routines are implemented in the repository's Python modules (`specialists/single_image/adaptation/qwen25vl/` and `specialists/single_image/training/colab/`).
- **Target Runtime**: Google Colab NVIDIA GPU (T4 / L4 / A100 with >= 15 GB VRAM).
- **Outputs**: Verifiable PEFT LoRA adapter (`adapter_model.safetensors`), training telemetry, and held-out evaluation reports.

## 1. Repository Setup & Clone

In [ ]:
# Clone or navigate to SatQuery repository
import os
if not os.path.exists("SatQuery"):
    !git clone -b feature/sruthi-single-image https://github.com/Lalith2007/SatQuery.git
    %cd SatQuery
else:
    %cd SatQuery
    !git checkout feature/sruthi-single-image
    !git pull


## 2. Dependency Installation (Reproducible Stack)

In [ ]:
!pip install --upgrade pip
!pip install pydantic>=2.6.0 pydantic-settings>=2.2.0 fastapi uvicorn pyyaml psutil tifffile httpx
!pip install transformers==5.16.1 datasets==5.0.1 peft==0.20.0 bitsandbytes==0.50.2 accelerate==1.14.0 trl==1.12.0 qwen-vl-utils==0.0.14 Pillow safetensors
!pip install -r requirements.txt || true
!python -c "import pydantic_settings, torch, transformers, peft, trl; print('All dependencies verified! CUDA:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


## 3. Step 00: Environment Diagnostics & Manifest Generation

In [ ]:
!python -m specialists.single_image.training.colab.00_environment_check

## 4. Step 01: Remote-Sensing Dataset Preparation
Generates canonical multimodal instruction samples (Optical + SAR) across VQA (35%), Grounding (30%), Captioning (25%), and Cross-Modal (10%) with 0 parent-scene spatial leakage.

In [ ]:
!python -m specialists.single_image.training.colab.01_prepare_dataset

## 5. Step 02: Strict Dataset Quality Validation Gate

In [ ]:
!python -m specialists.single_image.training.colab.02_validate_dataset

## 6. Step 03: Model Architecture & Tokenizer Inspection
Inspects Qwen2.5-VL module hierarchy, confirms `<|box_start|>` special tokens, and verifies LoRA target discovery (freezing vision backbone, adapting language decoder + visual merger).

In [ ]:
!python -m specialists.single_image.training.colab.03_inspect_qwen --model_id Qwen/Qwen2.5-VL-3B-Instruct

## 7. Step 04: Real CUDA Multimodal Smoke Test
Runs 5 real optical and SAR samples through collation, forward pass, loss computation, backward pass, optimizer step, and adapter save/reload.

In [ ]:
!python -m specialists.single_image.training.colab.04_smoke_test --model_id Qwen/Qwen2.5-VL-3B-Instruct

## 8. Step 05: Micro-Batch Overfit Verification
Optimizes 8 samples for 25 steps to verify that the multimodal loss drops by >25% and that native grounding syntax is learnable.

In [ ]:
!python -m specialists.single_image.training.colab.05_overfit_microbatch --model_id Qwen/Qwen2.5-VL-3B-Instruct --steps 25

## 9. Step 06: Full Production 4-Bit QLoRA Training
Launches TRL SFTTrainer with `paged_adamw_8bit`, gradient checkpointing, and cosine annealing.

In [ ]:
!python -m specialists.single_image.training.colab.06_train_qwen25vl_qlora --config configs/qwen25vl_qlora.yaml

## 10. Step 07: Held-Out Authoritative Evaluation
Evaluates the trained adapter on held-out test scenes for VQA accuracy, Grounding IoU & Recall@0.50/0.75, and Captioning.

In [ ]:
!python -m specialists.single_image.training.colab.07_evaluate_qwen25vl --test_file data/qwen_dataset/test.jsonl

## 11. Step 08 & 09: Adapter Checksum Verification & Provenance Packaging

In [ ]:
!python -m specialists.single_image.training.colab.08_export_adapter
!python -m specialists.single_image.training.colab.09_package_artifacts

## 12. Download / Commit Trained Adapter Artifacts
Once verified, the adapter files in `specialists/single_image/weights/qwen25vl_lora/` can be committed or exported back to the repository.

In [ ]:
!ls -lh specialists/single_image/weights/qwen25vl_lora/
# Optional: archive adapter directory for quick download
!tar -czvf qwen25vl_lora_weights.tar.gz specialists/single_image/weights/qwen25vl_lora/